In [95]:
from sklearn.datasets import make_classification
import torch

In [96]:
X, y = make_classification(
    n_classes=2,
    n_samples=100,
    n_features=3,
    n_redundant=0,
    n_informative=3,
    random_state=42
)

In [97]:
X.shape

(100, 3)

In [98]:
y

array([1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1,
       1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1,
       0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1])

In [99]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).view(-1,1)

In [100]:
from torch.utils.data import Dataset, DataLoader

In [101]:
# Dataset

class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [102]:
train_data = CustomDataset(X, y)

In [103]:
train_data_loader = DataLoader(train_data, batch_size=8, shuffle=True)

In [104]:
import torch.nn as nn

In [105]:
# model

class ANNClassifier(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 3),
            nn.ReLU(),
            nn.Linear(3,1),
        )

    def forward(self, features):
        return self.model(features)

In [106]:
classifier = ANNClassifier(X.shape[1])

In [107]:
from torchinfo import summary

summary(classifier, X.shape)

Layer (type:depth-idx)                   Output Shape              Param #
ANNClassifier                            [100, 1]                  --
├─Sequential: 1-1                        [100, 1]                  --
│    └─Linear: 2-1                       [100, 3]                  12
│    └─ReLU: 2-2                         [100, 3]                  --
│    └─Linear: 2-3                       [100, 1]                  4
Total params: 16
Trainable params: 16
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [108]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(classifier.parameters())

In [109]:
epochs = 100

for epoch in range(epochs):

    epoch_loss = 0.0
    running_epoch_loss = 0.0

    classifier.train()

    for xb, yb in train_data_loader:
        # forward pass
        y_pred = classifier(xb)

        # loss computation
        loss = criterion(y_pred, yb)

        # clear gradients to avoid accumulation
        optimizer.zero_grad()

        # backward pass
        loss.backward()

        # update parameters
        optimizer.step()

        running_epoch_loss += loss.item()

    epoch_loss = running_epoch_loss/len(train_data_loader)

    if epoch % 10 == 0 or (epoch+1)%10 == 0:
        print(f'{epoch+1}/{epochs} : loss = {epoch_loss}')


1/100 : loss = 0.6606420645347009
10/100 : loss = 0.5881306024698111
11/100 : loss = 0.5793023705482483
20/100 : loss = 0.5097187665792612
21/100 : loss = 0.5049079175178821
30/100 : loss = 0.4338542108352368
31/100 : loss = 0.4255700776210198
40/100 : loss = 0.3620799229695247
41/100 : loss = 0.36685434671548695
50/100 : loss = 0.32094984902785373
51/100 : loss = 0.3216111006645056
60/100 : loss = 0.2769159239072066
61/100 : loss = 0.2773226774655856
70/100 : loss = 0.2659781506428352
71/100 : loss = 0.26075477783496565
80/100 : loss = 0.24013965061077705
81/100 : loss = 0.24403184423079857
90/100 : loss = 0.23063458502292633
91/100 : loss = 0.22804087629685035
100/100 : loss = 0.21947428670067054


In [110]:
import torch
from sklearn.metrics import classification_report

all_preds = []
all_labels = []

with torch.no_grad():
    classifier.eval()

    for xb, yb in train_data_loader:
        # Forward pass
        logits = classifier(xb)
        
        # Binary classification thresholding
        preds = (torch.sigmoid(logits) > 0.5).int()
        
        # Flatten and store predictions/labels across all batches
        all_preds.extend(preds.cpu().flatten().numpy())
        all_labels.extend(yb.cpu().flatten().numpy())

# Generate and print the single comprehensive report
final_report = classification_report(all_labels, all_preds, zero_division=0)
print(final_report)


              precision    recall  f1-score   support

         0.0       0.94      0.90      0.92        50
         1.0       0.90      0.94      0.92        50

    accuracy                           0.92       100
   macro avg       0.92      0.92      0.92       100
weighted avg       0.92      0.92      0.92       100

